# 44. Fine-tuning, LoRA, distillation, and preference learning

**Level:** Advanced

**Prerequisites:** 27-28, 31, and 38: gradients, transfer, and language objectives.

[Course index](00_START_HERE.ipynb) | [Previous](43_LLM_Evaluation_Hallucinations_and_Reliability.ipynb) | [Next](45_Quantization_KV_Cache_and_Inference_Cost.ipynb)

**Topics:** Fine-tuning, SFT, LoRA, QLoRA, Distillation, RLHF, DPO, Post-training.

**Matching script:** `examples\44_FineTuning_LoRA_Distillation_and_Preferences.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Supervised fine-tuning trains on task or instruction examples after a base model is learned.
It can adapt format or behavior, but does not guarantee current facts and can degrade other
capabilities. Retrieval is often the right mechanism for changing reference knowledge.

LoRA freezes a base weight matrix and learns a low-rank update BA. It reduces trainable
parameter count, not necessarily all training memory or every compute cost. QLoRA combines
quantized base-model storage with adapter training and additional numerical/memory techniques;
merely rounding a matrix and adding LoRA is not a complete QLoRA implementation.

Distillation trains a student to match a teacher's outputs or distributions. Teacher errors,
licensing, data privacy, and distribution mismatch can transfer too. Temperature-softened
target distributions can contain more information than a single hard label.

RLHF is a broad pipeline using human preference information, often a learned reward model
and policy optimization. DPO optimizes a preference-derived objective directly from chosen/
rejected examples under its assumptions. They are not equivalent to tabular Q-learning.
Recent verifiable-reward training uses checkable outcomes where available, but reward hacking
and benchmark limitations remain concerns.

## Mathematics or data contract

For base W of shape (out,in), LoRA learns B(out,r) and A(r,in): W_eff = W + scaling*(B@A).

## Important limitation

The adapter is trained on a synthetic linear task; no pretrained LLM or full QLoRA/RLHF system is used.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Train a low-rank update while preserving the base

**Purpose:** Implement and test the central LoRA mechanism.

### Read the code in this order
1. Create a frozen base matrix and a known rank-one target change.
2. Optimize only A and B.
3. Compare initial and final reconstruction losses.
4. Assert the base stayed unchanged and adapter parameter count is smaller.

### Syntax and behavior to notice
- `backward()` computes/accumulates gradients through the recorded computation graph.
- `zero_grad()` clears earlier gradients before the next independent training update.
- `optimizer.step()` updates only the parameters owned by that optimizer.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
base/adapter parameters: 48 14
loss: 2.7958 -> 0.0
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import torch

torch.set_num_threads(1)
torch.manual_seed(42)
base = torch.randn(6, 8)
saved = base.clone()
X = torch.randn(100, 8)
true_update = torch.randn(6, 1) @ torch.randn(1, 8)
targets = X @ (base + true_update).T
A = torch.nn.Parameter(torch.randn(1, 8) * 0.1)
B = torch.nn.Parameter(torch.zeros(6, 1))
optimizer = torch.optim.Adam([A, B], lr=0.05)
initial = ((X @ base.T - targets) ** 2).mean().item()
for _ in range(180):
    optimizer.zero_grad()
    predictions = X @ (base + B @ A).T
    loss = ((predictions - targets) ** 2).mean()
    loss.backward()
    optimizer.step()
torch.testing.assert_close(base, saved)
assert loss.item() < initial * 0.01
assert A.numel() + B.numel() < base.numel()
print("base/adapter parameters:", base.numel(), A.numel() + B.numel())
print("loss:", round(initial, 4), "->", round(loss.item(), 4))

base/adapter parameters: 48 14
loss: 2.7958 -> 0.0


### Interpret example 1

The result should demonstrate this purpose: **Implement and test the central LoRA mechanism.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 2: Distill soft targets into a small student

**Purpose:** Actually optimize a student against teacher distributions instead of using distillation only as vocabulary.

### Read the code in this order
1. Define a frozen synthetic teacher mapping.
2. Calculate temperature-softened target probabilities.
3. Train a smaller student's logits with KL divergence.
4. Assert matching loss decreases and teacher state is unchanged.

### Syntax and behavior to notice
- `backward()` computes/accumulates gradients through the recorded computation graph.
- `zero_grad()` clears earlier gradients before the next independent training update.
- `optimizer.step()` updates only the parameters owned by that optimizer.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.
- `with`/`async with` bounds a resource lifetime or a temporary execution mode.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
synthetic teacher/student KL loss: 1.4498 -> 0.0
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [2]:
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)
X = torch.randn(100, 3)
teacher_weights = torch.tensor([[2., -1., 0.5], [-1., 2., -0.5]])
with torch.no_grad():
    teacher_logits = X @ teacher_weights.T
temperature = 2.0
teacher_probabilities = F.softmax(teacher_logits / temperature, dim=1)
student = nn.Linear(3, 2)
optimizer = torch.optim.Adam(student.parameters(), lr=0.05)
def distillation_loss():
    return F.kl_div(F.log_softmax(student(X) / temperature, dim=1),
                    teacher_probabilities, reduction="batchmean") * temperature**2
initial = distillation_loss().item()
for _ in range(100):
    optimizer.zero_grad()
    loss = distillation_loss()
    loss.backward()
    optimizer.step()
assert loss.item() < initial * 0.02
print("synthetic teacher/student KL loss:", round(initial, 4), "->", round(loss.item(), 4))

synthetic teacher/student KL loss: 1.4498 -> 0.0


### Interpret example 2

The result should demonstrate this purpose: **Actually optimize a student against teacher distributions instead of using distillation only as vocabulary.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 3: Inspect a pairwise preference objective

**Purpose:** Connect chosen/rejected probabilities to a simplified DPO loss calculation without claiming a full preference-training pipeline.

### Read the code in this order
1. Compute a policy chosen-minus-rejected log-probability ratio.
2. Subtract the frozen reference ratio.
3. Apply a stable negative log-sigmoid with a beta scale.
4. Check the loss is lower for a better preference margin.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
weak/improved pairwise loss: 0.7444 0.5981
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [3]:
import numpy as np

def pairwise_loss(policy_chosen, policy_rejected, reference_chosen, reference_rejected, beta=0.1):
    margin = beta * ((policy_chosen - policy_rejected) -
                     (reference_chosen - reference_rejected))
    return float(np.logaddexp(0., -margin))
weak = pairwise_loss(-3., -2., -2., -2.)
improved = pairwise_loss(-1., -3., -2., -2.)
assert improved < weak
assert np.isclose(pairwise_loss(-2., -2., -2., -2.), np.log(2))
print("weak/improved pairwise loss:", round(weak, 4), round(improved, 4))

weak/improved pairwise loss: 0.7444 0.5981


### Interpret example 3

The result should demonstrate this purpose: **Connect chosen/rejected probabilities to a simplified DPO loss calculation without claiming a full preference-training pipeline.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Vary adapter rank, count trainable parameters, and evaluate on held-out synthetic inputs.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Fine-tuning` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: For base W of shape (out,in), LoRA learns B(out,r) and A(r,in): W_eff = W + scaling*(B@A).
- Name the boundary of the demonstration: The adapter is trained on a synthetic linear task; no pretrained LLM or full QLoRA/RLHF system is used.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://arxiv.org/abs/2106.09685`
- `https://arxiv.org/abs/2305.14314`
- `https://arxiv.org/abs/2305.18290`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.